# PCA logit lens: unsupervised directions instead of a chosen contrast

The contrastive/"inverse" J-lens (`J_target − J_contrast`) came back null everywhere: cross-trait cosines were low, the one marginal panda finding (p=0.04) didn't survive residual removal, and logit-lensing `J_cat−dog` / `J_lion−dog` / `J_panda−dog` directly decoded to unrelated low-probability garbage (`top1_token_tables.ipynb`, section 3). That method is still SUPERVISED, though — it needs a human-chosen contrast token (`dog`) before it can run. This notebook drops that choice entirely: PCA directly on the raw activation/gradient distribution at `hidden_states[LAYER_SLOT]` (no labels, no chosen contrast), then logit-lens each of the top-N principal components — whatever directions the data's own variance actually concentrates on, regardless of what they turn out to mean. If a subliminal signal were a strong source of variance, an early PC should decode to something trait-relevant and/or align with the known `v_teacher` direction; if the top PCs are all generic and uncorrelated with `v_teacher`, that's another (this time unsupervised) angle coming back null.

In [1]:
import json
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

RLHF = Path('../rlhf').resolve()
STAGE1 = RLHF / 'stage1_subliminal_traits/runs/deepjudge_paper3'
STAGE3 = RLHF / 'stage3_eval_awareness_dpo/runs/eval_awareness_dpo_s1'
LAYER_SLOT = 11  # same convention as jlens_probe.py / gradient_probe.py / predictive_debug_probe.py
N_PCS = 20

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
DEVICE = 'cuda'

print(f'loading {MODEL_ID} on {DEVICE} -- run once, then skip this cell')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation='sdpa', device_map=DEVICE)
model.eval()
print('loaded.')


/home/chriskino/subliminal-learning-model-organism/rlhf/vendor/steering-vector-distillation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading Qwen/Qwen2.5-7B-Instruct on cuda -- run once, then skip this cell


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|                         | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|████▎            | 1/4 [00:00<00:02,  1.19it/s]

Loading checkpoint shards:  50%|████████▌        | 2/4 [00:01<00:01,  1.16it/s]

Loading checkpoint shards:  75%|████████████▊    | 3/4 [00:02<00:00,  1.15it/s]

Loading checkpoint shards: 100%|█████████████████| 4/4 [00:03<00:00,  1.22it/s]

Loading checkpoint shards: 100%|█████████████████| 4/4 [00:03<00:00,  1.20it/s]

loaded.


In [2]:
@torch.no_grad()
def pca_components(X: torch.Tensor, n: int):
    """X: [N, H]. Returns (components [n, H] unit vectors, explained_variance_ratio [n])."""
    mean = X.mean(0, keepdim=True)
    Xc = (X - mean).float()
    torch.manual_seed(0)  # pca_lowrank uses a random projection internally -- unseeded, low-variance
    # tail PCs (explained_var_% ~1%, a near-degenerate subspace) are NOT reproducible run-to-run;
    # seeding makes this notebook's own results stable, but doesn't make a low-variance PC's
    # direction any more meaningful -- see section 3's note on the alley/goose instability.
    U, S, V = torch.pca_lowrank(Xc, q=min(n + 5, Xc.shape[1]), niter=10)
    total_var = (Xc ** 2).sum() / (Xc.shape[0] - 1)
    var_per_pc = S[:n] ** 2 / (Xc.shape[0] - 1)
    explained = (var_per_pc / total_var).tolist()
    components = V[:, :n].T  # [n, H]
    return components, explained


@torch.no_grad()
def lens_direction_top1(direction: torch.Tensor) -> tuple[str, float]:
    h = direction.to(model.device, dtype=model.dtype)
    logits = model.lm_head(model.model.norm(h)).float()
    probs = logits.softmax(dim=-1)
    p, ix = probs.max(dim=-1)
    return tokenizer.decode([ix.item()]), p.item()


def pca_logit_lens_table(X: torch.Tensor, v_teacher: torch.Tensor | None, n: int = N_PCS) -> pd.DataFrame:
    components, explained = pca_components(X, n)
    rows = []
    for i in range(n):
        pc = components[i]
        tok_pos, p_pos = lens_direction_top1(pc)
        tok_neg, p_neg = lens_direction_top1(-pc)
        row = {
            'pc': i,
            'explained_var_%': round(explained[i] * 100, 2),
            'top1(+PC)': f'{tok_pos!r} ({p_pos:.2f})',
            'top1(-PC)': f'{tok_neg!r} ({p_neg:.2f})',
        }
        if v_teacher is not None:
            row['cos(PC, v_teacher)'] = round(F.cosine_similarity(pc.unsqueeze(0), v_teacher.unsqueeze(0)).item(), 4)
        rows.append(row)
    return pd.DataFrame(rows)


## 1. PCA on raw completion activations (cat / lion / panda)

`completion_activations_{trait}_n1024_seed0.pt` -- the SAME raw activation vectors `predictive_debug_probe.py`'s diff-in-means probe used (preferred + dispreferred, 2048 rows total per trait), just decomposed by unsupervised PCA instead of a supervised mean-difference. `cos(PC, v_teacher)` checks whether any of the top-20 PCs happens to align with the known, real trait direction even though PCA never saw it.

In [3]:
activation_pca_tables = {}
for trait in ['cat', 'lion', 'panda']:
    cache = torch.load(STAGE1 / 'vectors' / f'completion_activations_{trait}_n1024_seed0.pt', map_location='cpu', weights_only=False)
    X = torch.cat([cache['vecs_preferred'], cache['vecs_dispreferred']], dim=0)  # [2048, H]
    v_teacher = torch.load(STAGE1 / 'vectors' / f'v_teacher_{trait}.pt', map_location='cpu', weights_only=False)['raw'][LAYER_SLOT]
    activation_pca_tables[trait] = pca_logit_lens_table(X, v_teacher)
    top_cos = activation_pca_tables[trait]['cos(PC, v_teacher)'].abs().max()
    print(f'{trait}: top-20 PCs explain {activation_pca_tables[trait]["explained_var_%"].sum():.1f}% of variance; '
          f'max |cos(PC, v_teacher)| over top 20 = {top_cos:.4f}')


cat: top-20 PCs explain 76.4% of variance; max |cos(PC, v_teacher)| over top 20 = 0.0416


lion: top-20 PCs explain 76.3% of variance; max |cos(PC, v_teacher)| over top 20 = 0.0398


panda: top-20 PCs explain 76.3% of variance; max |cos(PC, v_teacher)| over top 20 = 0.0507


In [4]:
for trait, table in activation_pca_tables.items():
    print(f'=== {trait}: activation PCA ===')
    display(table)


=== cat: activation PCA ===


,pc,explained_var_%,top1(+PC),top1(-PC),"cos(PC, v_teacher)"
0,0,14.18,'.firebaseio' (0.05),'stituição' (0.20),-0.0416
1,1,10.56,'))?' (0.06),'perPage' (0.06),0.0035
2,2,9.40,'下面是小' (0.15),'ogenerated' (0.06),-0.0175
3,3,7.51,' ohio' (0.09),'рупп' (0.45),-0.0034
4,4,5.05,'ameleon' (0.04),' ' (0.11),0.0143
5,5,4.25,'stitución' (0.14),'osemite' (0.12),-0.0178
6,6,3.70,'ollapsed' (0.27),' validationResult' (0.10),-0.0030
7,7,3.29,'oriously' (0.22),'BASEPATH' (0.47),0.0049
8,8,2.87,'从根本' (0.13),'📐' (0.14),0.0167
9,9,2.53,' setBackgroundImage' (0.04),'comings' (0.68),0.0231


=== lion: activation PCA ===


,pc,explained_var_%,top1(+PC),top1(-PC),"cos(PC, v_teacher)"
0,0,14.22,'.firebaseio' (0.04),'stituição' (0.21),-0.0396
1,1,10.38,'))?' (0.07),' negó' (0.06),0.0055
2,2,9.52,'下面是小' (0.14),'ogenerated' (0.07),-0.0221
3,3,7.46,' ohio' (0.09),'рупп' (0.52),-0.0014
4,4,5.16,'ameleon' (0.05),' ' (0.13),0.0152
5,5,4.17,'stitución' (0.15),'osemite' (0.08),-0.0144
6,6,3.70,'soever' (0.09),'ollapsed' (0.21),-0.0003
7,7,3.29,'BASEPATH' (0.33),'oriously' (0.35),-0.0028
8,8,2.80,'lijah' (0.13),'=-=-=-=-' (0.11),0.0176
9,9,2.56,'comings' (0.50),' setBackgroundImage' (0.10),-0.0234


=== panda: activation PCA ===


,pc,explained_var_%,top1(+PC),top1(-PC),"cos(PC, v_teacher)"
0,0,14.00,'.firebaseio' (0.04),'stituição' (0.18),-0.0507
1,1,10.45,'))?' (0.06),'perPage' (0.06),0.0029
2,2,9.35,'下面是小' (0.14),'toi' (0.07),-0.0124
3,3,7.62,' bigot' (0.08),'рупп' (0.53),-0.0067
4,4,5.10,'왼' (0.04),' ' (0.08),0.0105
5,5,4.22,' Erot' (0.11),'osemite' (0.19),-0.0148
6,6,3.66,'ollapsed' (0.33),' validationResult' (0.10),-0.0010
7,7,3.31,'oriously' (0.33),'BASEPATH' (0.46),-0.0007
8,8,2.83,'📐' (0.12),'ustomed' (0.14),-0.0174
9,9,2.57,'comings' (0.71),' setBackgroundImage' (0.07),-0.0289


## 2. PCA on raw loss gradients (cat / eval_awareness)

Same treatment applied to `gradient_activations_{trait}_n1024_seed0.pt` (the per-row completion-loss gradients `gradient_probe.py` found null in a diff-in-means test, and section 2 of `top1_token_tables.ipynb` found null even after Adam-preconditioning). PCA here answers a different question than either of those: not "does the mean preferred-vs-dispreferred difference align with v_teacher" and not "does Adam-preconditioning change the mean direction" but "is there SOME direction in the raw gradient variance (not necessarily the mean, not necessarily Adam-scaled) that carries trait signal".

In [5]:
gradient_pca_tables = {}
for trait, run_dir in [('cat', STAGE1), ('eval_awareness', STAGE3)]:
    cache = torch.load(run_dir / 'vectors' / f'gradient_activations_{trait}_n1024_seed0.pt', map_location='cpu', weights_only=False)
    X = torch.cat([cache['grads_preferred'], cache['grads_dispreferred']], dim=0)  # [2048, H]
    v_teacher_path = run_dir / 'vectors' / f'v_teacher_{trait}.pt'
    v_teacher = torch.load(v_teacher_path, map_location='cpu', weights_only=False)['raw'][LAYER_SLOT] if v_teacher_path.exists() else None
    gradient_pca_tables[trait] = pca_logit_lens_table(X, v_teacher)
    top_cos = gradient_pca_tables[trait]['cos(PC, v_teacher)'].abs().max() if v_teacher is not None else float('nan')
    print(f'{trait}: top-20 PCs explain {gradient_pca_tables[trait]["explained_var_%"].sum():.1f}% of variance; '
          f'max |cos(PC, v_teacher)| over top 20 = {top_cos:.4f}')


cat: top-20 PCs explain 44.6% of variance; max |cos(PC, v_teacher)| over top 20 = 0.0361


eval_awareness: top-20 PCs explain 44.4% of variance; max |cos(PC, v_teacher)| over top 20 = 0.0432


In [6]:
for trait, table in gradient_pca_tables.items():
    print(f'=== {trait}: gradient PCA ===')
    display(table)
print(
    'Read: (1) do any top1(+-PC) tokens look trait-related (cat/animal words, or '
    "eval/monitor/test words for eval_awareness) rather than generic/garbage; "
    '(2) does cos(PC, v_teacher) spike meaningfully above the ~0 baseline for any PC, '
    'not just PC0. A high explained_var_% concentrated in PC0 with a near-zero cos to '
    'v_teacher would mean the dominant source of variance in this data is something '
    'else entirely (e.g. a generic outlier-feature direction, well documented in LLM '
    'activations generally) and unrelated to the trait -- consistent with every other '
    'unsupervised and supervised probe tried in this project.'
)


=== cat: gradient PCA ===


,pc,explained_var_%,top1(+PC),top1(-PC),"cos(PC, v_teacher)"
0,0,9.06,'intage' (0.04),' fick' (0.03),0.0162
1,1,6.10,'ihanna' (0.32),'prites' (0.03),0.0107
2,2,4.13,':frame' (0.06),'\ufeff/*\n' (0.03),-0.0081
3,3,3.17,'HideInInspector' (0.05),'ewire' (0.08),0.0007
4,4,2.75,' ula' (0.21),' Naughty' (0.07),-0.0072
5,5,2.32,'ienne' (0.02),'/filepath' (0.06),0.0139
6,6,1.88,' rencont' (0.14),'rlen' (0.38),0.0003
7,7,1.76,' IonicPage' (0.16),"'%;""' (0.18)",-0.0002
8,8,1.63,'不良信息' (0.07),'讧' (0.21),-0.0361
9,9,1.53,'stdafx' (0.04),'antro' (0.49),0.0031


=== eval_awareness: gradient PCA ===


,pc,explained_var_%,top1(+PC),top1(-PC),"cos(PC, v_teacher)"
0,0,7.98,'><?=$' (0.17),'iferay' (0.03),-0.0046
1,1,6.45,'℉' (0.26),'鸷' (0.16),-0.0050
2,2,5.14,'.getOwnProperty' (0.13),' ald' (0.02),0.0333
3,3,3.22,' Posté' (0.03),' 自动生成' (0.32),-0.0120
4,4,2.79,"""','');\n"" (0.16)",'琐' (0.34),-0.0034
5,5,2.04,'coni' (0.07),'在国外' (0.04),0.0094
6,6,1.87,' WHATSOEVER' (0.11),'eña' (0.23),-0.0178
7,7,1.72,'ALLOC' (0.05),' fkk' (0.21),0.0007
8,8,1.59,' Kimber' (0.04),"');"">\n' (0.17)",0.0352
9,9,1.42,"',"");\n' (0.13)",' Pitt' (0.08),-0.0051


Read: (1) do any top1(+-PC) tokens look trait-related (cat/animal words, or eval/monitor/test words for eval_awareness) rather than generic/garbage; (2) does cos(PC, v_teacher) spike meaningfully above the ~0 baseline for any PC, not just PC0. A high explained_var_% concentrated in PC0 with a near-zero cos to v_teacher would mean the dominant source of variance in this data is something else entirely (e.g. a generic outlier-feature direction, well documented in LLM activations generally) and unrelated to the trait -- consistent with every other unsupervised and supervised probe tried in this project.


## 3. Automated keyword scan (including idiom-adjacent words, e.g. "alley")

Eyeballing the tables above isn't reliable -- manually spotting `'alley'` in the eval_awareness gradient PCA table (PC15, −PC direction) is exactly the kind of thing a scan should catch instead of relying on a human noticing it. `alley` is worth including explicitly since "alley cat" is a common enough English idiom that the word carries real cat-association, not just literal `cat`/`kitten`/`feline`. Scans every `top1(+PC)`/`top1(-PC)` cell in every table built above (both activation-PCA and gradient-PCA, all traits) and reports every hit with its PC index, explained variance, and cosine to `v_teacher` -- so this can be judged on the actual numbers instead of which particular word happened to catch the eye.

**Important caveat found while adding this section**: `torch.pca_lowrank` uses an unseeded random projection internally. Before the fix above (setup cell now calls `torch.manual_seed(0)`), re-running section 2 produced a DIFFERENT `'alley'`-free set of top-1 decodes for PC10+ (`'goose'` and `'alley'` were both replaced by unrelated words like `/gtest` and `NotificationCenter` on rerun) -- i.e. the words that first caught the eye were themselves an artifact of unseeded randomness in a near-degenerate, low-variance subspace (PC10+ explain ~1% of variance each), not a stable feature of the data. That instability is itself informative: a real signal wouldn't disappear on a re-run with the same data. The run below is seeded for reproducibility, but a seeded-but-still-low-variance PC is not thereby more meaningful -- only more reproducible.

In [7]:
import re

KEYWORDS = ['cat', 'kitten', 'feline', 'meow', 'whisker', 'paw', 'alley', 'tabby', 'tomcat',
            'lion', 'panda', 'dog', 'puppy', 'canine', 'bamboo',
            'eval', 'test', 'monitor', 'judge', 'score', 'aware', 'exam']
_kw_re = re.compile('|'.join(re.escape(k) for k in KEYWORDS), re.IGNORECASE)

def scan_pca_table(name, df):
    hits = []
    for col in ['top1(+PC)', 'top1(-PC)']:
        for pc_idx, val in zip(df['pc'], df[col]):
            s = str(val)
            m = re.search(r"'([^']*)'", s)
            tok = m.group(1) if m else s
            if _kw_re.search(tok):
                row = df[df['pc'] == pc_idx].iloc[0]
                hits.append({
                    'source': name, 'pc': pc_idx, 'direction': col,
                    'matched_token': tok, 'full_cell': s,
                    'explained_var_%': row['explained_var_%'],
                    'cos(PC, v_teacher)': row.get('cos(PC, v_teacher)', float('nan')),
                })
    return hits

all_pca_hits = []
for trait, df in activation_pca_tables.items():
    all_pca_hits += scan_pca_table(f'activation_pca[{trait}]', df)
for trait, df in gradient_pca_tables.items():
    all_pca_hits += scan_pca_table(f'gradient_pca[{trait}]', df)

if all_pca_hits:
    display(pd.DataFrame(all_pca_hits))
else:
    print(f'No hits for any of {KEYWORDS}.')
print(f'Total hits: {len(all_pca_hits)}  (scanned {len(activation_pca_tables) + len(gradient_pca_tables)} tables x 20 PCs x 2 directions = '
      f'{(len(activation_pca_tables) + len(gradient_pca_tables)) * 20 * 2} cells)')


,source,pc,direction,matched_token,full_cell,explained_var_%,"cos(PC, v_teacher)"
0,gradient_pca[eval_awareness],10,top1(+PC),/gtest,'/gtest' (0.09),1.39,0.0211
1,gradient_pca[eval_awareness],12,top1(-PC),NotificationCenter,' NotificationCenter' (0.08),1.10,0.0316


Total hits: 2  (scanned 5 tables x 20 PCs x 2 directions = 200 cells)
